In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/defog/sqlcoder-7b

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/defog/sqlcoder-7b)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="defog/sqlcoder-7b")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

Device set to use cuda:0


In [16]:
PROMPT_TEMPLATE = """### Task
Generate a SQL query to answer [QUESTION]{user_question}[/QUESTION]

### Database Schema
The query will run on a database with the following schema:
{data_dim_model}

### Answer
Given the database schema, here is the SQL query that [QUESTION]{user_question}[/QUESTION]
[SQL]
"""
DECODE_CONFIG = dict(do_sample=False, num_beams=4)
DEMO_SCHEMA_PGSQL = """
CREATE TABLE organisations (
  organisation_id      BIGINT PRIMARY KEY,
  organisation_name    TEXT NOT NULL,
  industry             TEXT,
  country_code         CHAR(2),
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE users (
  user_id              BIGINT PRIMARY KEY,
  organisation_id      BIGINT NOT NULL REFERENCES organisations(organisation_id),
  email                TEXT NOT NULL UNIQUE,
  full_name            TEXT,
  role                 TEXT,
  is_active            BOOLEAN NOT NULL DEFAULT TRUE,
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE applications (
  application_id       BIGINT PRIMARY KEY,
  organisation_id      BIGINT NOT NULL REFERENCES organisations(organisation_id),
  broker_user_id       BIGINT REFERENCES users(user_id),
  lender_user_id       BIGINT REFERENCES users(user_id),
  stage                TEXT NOT NULL, -- e.g. 'new','submitted','approved','settled','declined'
  status               TEXT NOT NULL, -- e.g. 'open','closed'
  submitted_at         TIMESTAMP,
  decisioned_at        TIMESTAMP,
  settled_at           TIMESTAMP,
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE borrowers (
  borrower_id          BIGINT PRIMARY KEY,
  application_id       BIGINT NOT NULL REFERENCES applications(application_id),
  borrower_type        TEXT NOT NULL, -- 'individual' | 'company'
  first_name           TEXT,
  last_name            TEXT,
  company_name         TEXT,
  abn                  TEXT,
  date_of_birth        DATE,
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE assets (
  asset_id             BIGINT PRIMARY KEY,
  application_id       BIGINT NOT NULL REFERENCES applications(application_id),
  asset_type           TEXT NOT NULL, -- 'vehicle','equipment','property','other'
  make                 TEXT,
  model                TEXT,
  year                 INT,
  purchase_price       NUMERIC(18,2),
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE loan_details (
  loan_id              BIGINT PRIMARY KEY,
  application_id       BIGINT NOT NULL UNIQUE REFERENCES applications(application_id),
  requested_amount     NUMERIC(18,2) NOT NULL,
  approved_amount      NUMERIC(18,2),
  interest_rate_pct    NUMERIC(6,3),
  term_months          INT,
  repayment_type       TEXT,          -- 'PI','IO'
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE application_events (
  event_id             BIGINT PRIMARY KEY,
  application_id       BIGINT NOT NULL REFERENCES applications(application_id),
  event_type           TEXT NOT NULL, -- 'created','submitted','docs_received','approved','declined','settled'
  event_time           TIMESTAMP NOT NULL,
  actor_user_id        BIGINT REFERENCES users(user_id),
  notes                TEXT
);

CREATE TABLE payments (
  payment_id           BIGINT PRIMARY KEY,
  application_id       BIGINT NOT NULL REFERENCES applications(application_id),
  payment_date         DATE NOT NULL,
  amount               NUMERIC(18,2) NOT NULL,
  payment_status       TEXT NOT NULL, -- 'paid','failed','pending'
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE credit_scores (
  score_id             BIGINT PRIMARY KEY,
  application_id       BIGINT NOT NULL REFERENCES applications(application_id),
  bureau               TEXT NOT NULL, -- 'equifax','experian','illion'
  score_value          INT NOT NULL,
  score_time           TIMESTAMP NOT NULL
);

-- Helpful indexes (optional metadata)
CREATE INDEX idx_applications_org_created_at ON applications(organisation_id, created_at);
CREATE INDEX idx_events_app_time ON application_events(application_id, event_time);
CREATE INDEX idx_payments_app_date ON payments(application_id, payment_date);"""
DEMO_INPUT_QUESTION = "What is the monthly approval rate and average approved amount for each organisation in 2025?"
inputs = PROMPT_TEMPLATE.format(user_question=DEMO_INPUT_QUESTION, data_dim_model=DEMO_SCHEMA_PGSQL)

In [17]:
output = pipe(inputs, do_sample=False, num_beams=1)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [18]:
print(output[0].get("generated_text"))

### Task
Generate a SQL query to answer [QUESTION]What is the monthly approval rate and average approved amount for each organisation in 2025?[/QUESTION]

### Database Schema
The query will run on a database with the following schema:

CREATE TABLE organisations (
  organisation_id      BIGINT PRIMARY KEY,
  organisation_name    TEXT NOT NULL,
  industry             TEXT,
  country_code         CHAR(2),
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE users (
  user_id              BIGINT PRIMARY KEY,
  organisation_id      BIGINT NOT NULL REFERENCES organisations(organisation_id),
  email                TEXT NOT NULL UNIQUE,
  full_name            TEXT,
  role                 TEXT,
  is_active            BOOLEAN NOT NULL DEFAULT TRUE,
  created_at           TIMESTAMP NOT NULL
);

CREATE TABLE applications (
  application_id       BIGINT PRIMARY KEY,
  organisation_id      BIGINT NOT NULL REFERENCES organisations(organisation_id),
  broker_user_id       BIGINT REFERENCES user